# ¿Qué hogares presentan inseguridad alimentaria?
**Taller práctico — ENSANUT 2025**

Describimos los hogares, entrenamos un árbol de decisión y medimos cuánto acierta. Al final quedan otros algoritmos para practicar en casa.

## 1. Cargar los datos

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
URL_DATOS = "https://raw.githubusercontent.com/jdelacruzm/AnalisisDatos/main/ensanut2025_hogares.csv"
hogares = pd.read_csv(URL_DATOS)
print("Hogares encuestados:", len(hogares))

In [ ]:
display(hogares.head())

## 2. Explorar la inseguridad alimentaria
Los porcentajes se calculan con el factor de expansión: cada hogar encuestado representa a muchos hogares del país.

In [ ]:
niveles = ["Seguridad", "Leve", "Moderada", "Severa"]
porcentajes = hogares.groupby("inseguridad_alimentaria")["factor_expansion"].sum()
porcentajes = 100 * porcentajes / porcentajes.sum()
display(porcentajes.reindex(niveles).round(1).rename("Hogares (%)"))

In [ ]:
def porcentaje_ponderado(datos):
    afectados = datos["inseguridad_moderada_o_severa"] == "Sí"
    return 100 * datos.loc[afectados, "factor_expansion"].sum() / datos["factor_expansion"].sum()

print(f"Inseguridad moderada o severa: {porcentaje_ponderado(hogares):.1f}%")

### Comparar grupos de hogares

In [ ]:
factores = ["ambito", "jefatura", "hay_menores", "nivel_socioeconomico",
            "recibe_programa_social", "escolaridad_jefe"]

In [ ]:
tablas = {}
for factor in factores:
    tablas[factor] = hogares.groupby(factor).apply(porcentaje_ponderado, include_groups=False).round(1)
    display(tablas[factor].rename("Inseguridad moderada o severa (%)"))

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(14, 4))
for eje, factor in zip(ejes, ["nivel_socioeconomico", "ambito", "jefatura"]):
    tablas[factor].sort_values().plot.barh(ax=eje, title=factor)
    eje.set_xlabel("Porcentaje")
plt.tight_layout()
plt.show()

## 3. Árbol de decisión
Los factores (`X`) se pasan a forma numérica; el resultado (`y`) es si el hogar presenta inseguridad moderada o severa.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree

In [ ]:
orden_escolaridad = ["sin escolaridad", "primaria o menos", "secundaria",
                     "preparatoria", "licenciatura o más"]

X = pd.DataFrame({
    "Hogar rural": (hogares["ambito"] == "Rural").astype(int),
    "Jefatura femenina": (hogares["jefatura"] == "Femenina").astype(int),
    "Integrantes": hogares["integrantes"],
    "Hay menores": (hogares["hay_menores"] == "Sí").astype(int),
    "Nivel socioeconómico bajo": hogares["nivel_socioeconomico"].str[0].astype(int).le(2).astype(int),
    "Recibe programa social": (hogares["recibe_programa_social"] == "Sí").astype(int),
    "Escolaridad del jefe": hogares["escolaridad_jefe"].str.lower().map(
        {valor: i for i, valor in enumerate(orden_escolaridad)}),
})
display(X.head())

In [ ]:
y = (hogares["inseguridad_moderada_o_severa"] == "Sí").astype(int)
pesos = hogares["factor_expansion"]

### Separar hogares para entrenar y para probar
70 de cada 100 hogares sirven para que el árbol aprenda; los otros 30 solo para comprobar cuánto acierta con hogares que no vio.

In [ ]:
X_entrena, X_prueba, y_entrena, y_prueba, w_entrena, w_prueba = train_test_split(
    X, y, pesos, test_size=0.30, random_state=2025, stratify=y
)
print("Entrenamiento:", len(X_entrena), "| Prueba:", len(X_prueba))

### Entrenar y dibujar el árbol

In [ ]:
arbol = DecisionTreeClassifier(max_depth=3, min_samples_leaf=120,
                               class_weight="balanced", random_state=2025)
arbol.fit(X_entrena, y_entrena, sample_weight=w_entrena)

In [ ]:
plt.figure(figsize=(20, 9))
plot_tree(arbol, feature_names=X.columns, class_names=["No", "Sí"], node_ids=True,
          impurity=False, proportion=True, rounded=True, fontsize=10)
plt.show()

### Perfiles que encuentra el árbol
Cada hoja es un grupo de hogares; su número (`node #`) aparece en el dibujo. Aquí se mide, con los hogares de prueba, qué porcentaje de cada hoja presenta inseguridad alimentaria.

In [ ]:
perfiles = pd.DataFrame({
    "Hoja": arbol.apply(X_prueba),
    "Inseguridad": y_prueba.to_numpy(),
    "Peso": w_prueba.to_numpy(),
})

In [ ]:
def resumen_de_hoja(grupo):
    afectados = grupo["Inseguridad"] == 1
    return pd.Series({
        "Hogares de prueba": len(grupo),
        "Inseguridad (%)": 100 * grupo.loc[afectados, "Peso"].sum() / grupo["Peso"].sum(),
    })

resumen = perfiles.groupby("Hoja").apply(resumen_de_hoja, include_groups=False)
resumen["Hogares de prueba"] = resumen["Hogares de prueba"].astype(int)
display(resumen.sort_values("Inseguridad (%)", ascending=False).round(1))

## 4. Evaluar el modelo
La matriz de confusión muestra los aciertos y errores del árbol con los hogares de prueba. Filas: la realidad; columnas: lo que dijo el modelo.

In [ ]:
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, recall_score, precision_score,
                             f1_score, balanced_accuracy_score, roc_auc_score)

In [ ]:
predicciones = arbol.predict(X_prueba)
ConfusionMatrixDisplay.from_predictions(y_prueba, predicciones, display_labels=["No", "Sí"])
plt.show()

In [ ]:
metricas = pd.Series({
    "Exactitud (accuracy)": accuracy_score(y_prueba, predicciones),
    "Sensibilidad (recall)": recall_score(y_prueba, predicciones),
    "Precisión (precision)": precision_score(y_prueba, predicciones),
    "Especificidad (specificity)": recall_score(y_prueba, predicciones, pos_label=0),
    "F1": f1_score(y_prueba, predicciones),
    "Exactitud balanceada (balanced accuracy)": balanced_accuracy_score(y_prueba, predicciones),
    "AUC": roc_auc_score(y_prueba, arbol.predict_proba(X_prueba)[:, 1]),
}, name="Valor")
display(metricas.round(2))

### ¿Qué factores utiliza más el árbol?
La importancia dice cuánto sirvió cada factor para separar a los hogares; no demuestra causalidad.

In [ ]:
importancia = pd.Series(arbol.feature_importances_, index=X.columns)
importancia.sort_values().plot.barh(title="Importancia de los factores")
plt.show()

---
## Para practicar en casa: otros algoritmos
Los mismos hogares de entrenamiento y de prueba, las mismas métricas; cambia el algoritmo. Los resultados se comparan, no se interpretan como relaciones causales.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
modelos = {
    "Árbol": arbol,
    "Regresión logística": make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000)),
    "Random Forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=50, class_weight="balanced", random_state=2025),
    "Gradient Boosting": HistGradientBoostingClassifier(max_depth=3, class_weight="balanced", random_state=2025),
    "SVM": make_pipeline(StandardScaler(), SVC(class_weight="balanced", random_state=2025)),
}

In [ ]:
resultados = []
for nombre, modelo in modelos.items():
    if nombre != "Árbol":
        if isinstance(modelo, Pipeline):
            paso = modelo.steps[-1][0]
            modelo.fit(X_entrena, y_entrena, **{f"{paso}__sample_weight": w_entrena})
        else:
            modelo.fit(X_entrena, y_entrena, sample_weight=w_entrena)
    pred = modelo.predict(X_prueba)
    puntaje = modelo.decision_function(X_prueba) if hasattr(modelo, "decision_function") else modelo.predict_proba(X_prueba)[:, 1]
    resultados.append({
        "Modelo": nombre,
        "Exactitud": accuracy_score(y_prueba, pred),
        "Sensibilidad": recall_score(y_prueba, pred),
        "Precisión": precision_score(y_prueba, pred),
        "F1": f1_score(y_prueba, pred),
        "AUC": roc_auc_score(y_prueba, puntaje),
    })

display(pd.DataFrame(resultados).round(2))

## Para practicar en casa (2): un factor trampa
Agregamos un factor que suena legítimo: "algún adulto del hogar dejó de comer por falta de dinero". Es una de las 15 preguntas con las que se construye la propia escala ELCSA, así que ya contiene la respuesta (*data leakage*). Observa cuánto "mejora" el modelo.

In [ ]:
X_trampa = X.copy()
X_trampa["Un adulto dejó de comer"] = (hogares["adulto_dejo_de_comer"] == "Sí").astype(int)
Xt_entrena, Xt_prueba = train_test_split(X_trampa, test_size=0.30, random_state=2025, stratify=y)

In [ ]:
arbol_trampa = DecisionTreeClassifier(max_depth=3, min_samples_leaf=120,
                                      class_weight="balanced", random_state=2025)
arbol_trampa.fit(Xt_entrena, y_entrena, sample_weight=w_entrena)
pred_trampa = arbol_trampa.predict(Xt_prueba)
print("Exactitud:", round(accuracy_score(y_prueba, pred_trampa), 2),
      "| Precisión:", round(precision_score(y_prueba, pred_trampa), 2),
      "| AUC:", round(roc_auc_score(y_prueba, arbol_trampa.predict_proba(Xt_prueba)[:, 1]), 2))

In [ ]:
plt.figure(figsize=(20, 9))
plot_tree(arbol_trampa, feature_names=X_trampa.columns, class_names=["No", "Sí"], node_ids=True,
          impurity=False, proportion=True, rounded=True, fontsize=10)
plt.show()

## Discusión final
- ¿Qué grupos presentan mayor inseguridad alimentaria?
- ¿Qué factores aparecen en las primeras divisiones del árbol?
- ¿Qué significa que el modelo no detecte a un hogar afectado?
- ¿Qué resultados sería necesario contrastar con la teoría y el trabajo de campo?

Puedes pedir a una IA que **interprete los resultados**, pero verifica sus afirmaciones con el análisis y la documentación de la encuesta.